# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"\nSchema URL: {croissant_url}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This helps you identify which record sets and fields you want to work with. All entities are referenced by their `@id`.

In [ ]:
# Display the available record sets by @id
record_sets = dataset.record_sets()
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

# For each record set, list its fields, columns, and their @ids
print("\nFields and columns for each record set:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        # f can be a plain string (the @id) or dict
        field_id = f if isinstance(f, str) else f.get('@id', '')
        print(f"    Field @id: {field_id}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for c in columns:
        column_id = c if isinstance(c, str) else c.get('@id', '')
        print(f"    Column @id: {column_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Select all available record set @id's and load each into a DataFrame
# Hint: Adjust this list based on the output from the previous cell if you want fewer record sets
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for rset_id in all_record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded {len(df)} rows for record set: {rset_id}")
    else:
        print(f"No records found for record set: {rset_id}")

# Choose one record set with data for further exploration
record_set_id = None
for rsid, _df in dataframes.items():
    record_set_id = rsid
    break
if record_set_id is None:
    raise ValueError("No record set with data found.")

print(f"\nExample record set for further analysis: '{record_set_id}'")
print("Fields (columns):", dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by categorical variables.

> **Note:** Replace `<numeric_field_id>` and `<group_field>` with actual `@id`s from dataframes' columns (see previous cell's output).

In [ ]:
# Identify numeric and categorical fields
df = dataframes[record_set_id]

# Suggest example numeric and group fields according to common mental health survey variables
numeric_candidates = [c for c in df.columns if 'score' in c.lower() or 'age' in c.lower() or df[c].dtype in [int, float]]
print("Numeric candidate columns:", numeric_candidates)

# Use the first numeric column for demonstration
numeric_field = numeric_candidates[0] if len(numeric_candidates) else None
print(f"Selected numeric field: {numeric_field}")

# Suggest grouping field candidates
group_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() > 1 and df[c].nunique() < 20]
print("Group-by candidate columns:", group_candidates)
group_field = group_candidates[0] if group_candidates else None
print(f"Selected group field: {group_field}\n")

if numeric_field is not None:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nMean {numeric_field} by {group_field} (for filtered records):")
        print(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    # Numeric field distribution
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        # Boxplot by group field
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use `mlcroissant` to discover and explore a FAIR² mental health survey dataset. We:
- Loaded and inspected Croissant metadata and the available record sets and fields by their `@id`.
- Extracted record set data and performed basic data analysis (e.g., normalization and grouping).
- Produced visualizations to support exploratory analysis.

**Next steps:** Use your domain knowledge of mental health or survey data to perform deeper analysis, select other record sets, or combine fields as appropriate.

For more information, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) or the dataset documentation linked from the Croissant schema.